In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

import multiprocessing as mp
from tqdm import tqdm
from joblib import Parallel, delayed
import os
from sklearn.feature_selection import mutual_info_classif
from stoch_sim_model import *

plt.style.use('/Users/ObinnaUkogu/Desktop/custom.mplstyle')
%config InlineBackend.figure_format = 'retina'

In [2]:
# Make grid
b_a1 = 2
t1 = 10
d_a1 = 3.8*10**(-4)
a1_0 = 100

cv_param = 0.1
runs = 10_000
MI_args = np.hstack(([2],np.arange(4,len(stat_names)))) # modify: 0 = a1_0, 1 = b_a1, 2 = t1

def comp_MI(param, cv_b_E,cv_g_ME, cv_b_c1 = 0):

    params = np.random.lognormal(mean = np.log(param/np.sqrt(1 + cv_param**2)), 
                            sigma = np.sqrt(np.log(1 + cv_param**2)), size = runs)
    
    # choose which parameter to vary
    run_data = np.array(Parallel(n_jobs=os.cpu_count(), batch_size=int(len(params)/8))(delayed(sum_sim)(t1 = param, noise_cv = [cv_b_E,cv_b_c1,cv_g_ME,0,0,0]) \
                                                    for param in params))
    
    MI_data = np.zeros(len(MI_args))
    for i in np.arange(len(MI_data)):
        MI_data[i] = calc_MI(run_data[:,2],run_data[:,MI_args[i]]) # modify: 0 = a1_0, 1 = b_a1, 2 = t1
        
    return  np.array(MI_data, dtype=object)

In [ ]:
# generate 2 2d grids for the b_E & g_ME bounds
nx, ny = 10, 10
cv_b_E, cv_g_ME = np.meshgrid(np.linspace(0, 1, nx), np.linspace(0, 1, ny))

v_comp_MI = np.vectorize(comp_MI)
MI_data_grid = v_comp_MI(b_a1, cv_b_E, cv_g_ME)

# function to slice data by parameters
extract_stats = np.vectorize(lambda x,stat: x[stat])

In [ ]:
# plot surface maps
fig = plt.figure(figsize=(12,12),dpi = 300)

i = 4

for val in stat_names[i:]:
    z = extract_stats(MI_data_grid,i-3)

    ax = fig.add_subplot(4, 2, i-3, projection='3d')
    cs = ax.plot_surface(cv_b_E, cv_g_ME, z, cmap = plt.cm.cividis)

    #ax.set_title(val, fontsize = 12)
    # set the limits of the plot to the limits of the data
    ax.axis([cv_g_ME.min(), cv_g_ME.max(), cv_g_ME.min(), cv_g_ME.max()])
    ax.set(xlabel=r'CV of $b_E$', ylabel=r'CV of $g_{ME}$')
    fig.colorbar(cs,  shrink=0.5, aspect=8, label =r"I("+val+r";$t_1)$")

    i += 1
    
fig.suptitle(r'$I(\cdot,t_1)$ as a function of noise in $b_{E}$ and $g_{ME}$', fontsize = 16)
fig.savefig('surf-MI-t1-vs-noise-b_E-g_ME.pdf', dpi=300, bbox_inches="tight")
plt.tight_layout()

plt.show()

In [ ]:
# plot heat maps
fig_b_a1, axs = plt.subplots(3, 2, dpi = 150, constrained_layout = True, figsize = (6,8))
i = 4

for ax, val in zip(axs.flat,stat_names[i:]):
    z = extract_stats(MI_data_grid,i-3)
    
    z = z[:-1, :-1]
    z_min, z_max = np.abs(z).min(), np.abs(z).max()

    cs = ax.pcolormesh(cv_b_E, cv_g_ME, z, cmap='RdBu', vmin=z_min, vmax=z_max)
    
    ax.set_title(val)
    # set the limits of the plot to the limits of the data
    ax.axis([cv_b_E.min(), cv_b_E.max(), cv_g_ME.min(), cv_g_ME.max()])
    ax.set(xlabel=r'CV of $b_E$', ylabel=r'CV of $g_{ME}$')
    fig_b_a1.colorbar(cs, ax=ax)
        
    i += 1
    
fig_b_a1.suptitle(r'$I(\cdot,t_1)$ as a function of population noise')
fig_b_a1.savefig('hmap-MI-vs_noise_t1.pdf', dpi=300, bbox_inches="tight")

